In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from src.custom_fastkan import FastKAN
import pandas as pd

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
def true_function(x):
    # f(x) = exp(1/100 * sum_{i=1}^{100} sin^2(pi * x_i / 2))
    # x is shape (N, 100)
    term = torch.sin(torch.pi * x / 2)**2
    return torch.exp(torch.mean(term, dim=1))

torch.manual_seed(42)
num_samples = 10000

# 100 dimensions
X = torch.rand(num_samples, 100) * 2 - 1 
y = true_function(X)

print(f"Input X range: min={X.min().item():.6f}, max={X.max().item():.6f}")

train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)
test_size = num_samples - train_size - val_size

X_train, X_val, X_test = torch.split(X, [train_size, val_size, test_size])
y_train, y_val, y_test = torch.split(y, [train_size, val_size, test_size])

X_train = X_train.to(device)
y_train = y_train.to(device)
X_val = X_val.to(device)
y_val = y_val.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Input X range: min=-1.000000, max=0.999997
Train shape: torch.Size([7000, 100]), Val shape: torch.Size([1500, 100]), Test shape: torch.Size([1500, 100])


In [3]:
# Use a slightly larger grid range to avoid boundary issues
model = FastKAN([100, 1, 1], grid_min=-1.2, grid_max=1.2, num_grids=10, use_base_update=False, use_layernorm=False).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [4]:
print("Training KAN...")
train_losses = []
val_losses = []

for epoch in range(1000):
    optimizer.zero_grad()
    pred = model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        with torch.no_grad():
            val_pred = model(X_val).squeeze()
            val_loss = torch.mean((val_pred - y_val)**2)
            train_losses.append(loss.item())
            val_losses.append(val_loss.item())
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

model.eval()
with torch.no_grad():
    test_pred = model(X_test).squeeze()
    mse = mean_squared_error(y_test.cpu(), test_pred.cpu())
    r2 = r2_score(y_test.cpu(), test_pred.cpu())
    
    print(f"KAN Test MSE: {mse:.6f}")
    print(f"KAN Test R2: {r2:.4f}")

Training KAN...
Epoch 0, Train MSE: 2.699224, Val MSE: 2.668748
Epoch 50, Train MSE: 0.421075, Val MSE: 0.412292
Epoch 100, Train MSE: 0.008338, Val MSE: 0.007494
Epoch 150, Train MSE: 0.003427, Val MSE: 0.004268
Epoch 200, Train MSE: 0.002793, Val MSE: 0.003686
Epoch 250, Train MSE: 0.002353, Val MSE: 0.003190
Epoch 300, Train MSE: 0.002017, Val MSE: 0.002785
Epoch 350, Train MSE: 0.001743, Val MSE: 0.002443
Epoch 400, Train MSE: 0.001514, Val MSE: 0.002153
Epoch 450, Train MSE: 0.001321, Val MSE: 0.001903
Epoch 500, Train MSE: 0.001156, Val MSE: 0.001683
Epoch 550, Train MSE: 0.001014, Val MSE: 0.001488
Epoch 600, Train MSE: 0.000893, Val MSE: 0.001317
Epoch 650, Train MSE: 0.000789, Val MSE: 0.001169
Epoch 700, Train MSE: 0.000700, Val MSE: 0.001041
Epoch 750, Train MSE: 0.000625, Val MSE: 0.000931
Epoch 800, Train MSE: 0.000560, Val MSE: 0.000836
Epoch 850, Train MSE: 0.000505, Val MSE: 0.000754
Epoch 900, Train MSE: 0.000457, Val MSE: 0.000681
Epoch 950, Train MSE: 0.000414, Val M

In [5]:
import torch.nn as nn
torch.manual_seed(42)

class MLP(nn.Module):
    def __init__(self, input_dim=100, hidden_dims=[128, 128, 128], output_dim=1, dropout_prob=0.0):
        super(MLP, self).__init__()
        layers = []
        curr_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(curr_dim, h_dim))
            layers.append(nn.GELU()) 
            if dropout_prob > 0:
                layers.append(nn.Dropout(dropout_prob))
            curr_dim = h_dim
        layers.append(nn.Linear(curr_dim, output_dim))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

# Very large capacity to match KAN's expressivity for this function
mlp_model = MLP(input_dim=100, hidden_dims=[1024, 1024, 1024], output_dim=1, dropout_prob=0.0).to(device)
mlp_optimizer = torch.optim.AdamW(mlp_model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(mlp_optimizer, mode='min', factor=0.5, patience=40, verbose=True)

print("Training MLP...")
best_val_loss = float('inf')
best_model_state = None

for epoch in range(2000): 
    mlp_model.train()
    mlp_optimizer.zero_grad()
    pred = mlp_model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    mlp_optimizer.step()
    
    # Validation
    mlp_model.eval()
    with torch.no_grad():
        val_pred = mlp_model(X_val).squeeze()
        val_loss = torch.mean((val_pred - y_val)**2)
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = mlp_model.state_dict()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}, LR: {mlp_optimizer.param_groups[0]['lr']:.6f}")

# Load best model
mlp_model.load_state_dict(best_model_state)

mlp_model.eval()
with torch.no_grad():
    mlp_pred = mlp_model(X_test).squeeze()
    mlp_mse = mean_squared_error(y_test.cpu(), mlp_pred.cpu())
    mlp_r2 = r2_score(y_test.cpu(), mlp_pred.cpu())
    
    print(f"\nMLP Test MSE: {mlp_mse:.6f}")
    print(f"MLP Test R2: {mlp_r2:.4f}")

Training MLP...


/Users/chandrakanthn/miniforge3/envs/jax-env/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 0, Train MSE: 2.740913, Val MSE: 2.026040, LR: 0.001000
Epoch 100, Train MSE: 0.000689, Val MSE: 0.000832, LR: 0.001000
Epoch 200, Train MSE: 0.000428, Val MSE: 0.000704, LR: 0.001000
Epoch 300, Train MSE: 0.000294, Val MSE: 0.000646, LR: 0.001000
Epoch 400, Train MSE: 0.000227, Val MSE: 0.000626, LR: 0.001000
Epoch 500, Train MSE: 0.000195, Val MSE: 0.000627, LR: 0.000500
Epoch 600, Train MSE: 0.000188, Val MSE: 0.000629, LR: 0.000063
Epoch 700, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000016
Epoch 800, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000002
Epoch 900, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000000
Epoch 1000, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000000
Epoch 1100, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000000
Epoch 1200, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000000
Epoch 1300, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000000
Epoch 1400, Train MSE: 0.000187, Val MSE: 0.000630, LR: 0.000000
Epoch 1500, Train MSE: 0.000187, Val 

In [6]:
print("\nAnalyzing individual KAN prediction losses...")
individual_losses = []
predictions = []
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        individual_losses.append(loss)
        predictions.append(prediction)

individual_losses = np.array(individual_losses)
predictions = torch.stack(predictions)
sorted_indices = np.argsort(individual_losses)

mean_loss = np.mean(individual_losses)
lowest_indices = sorted_indices[:3] 
highest_indices = sorted_indices[-3:]

mean_distances = np.abs(individual_losses - mean_loss)
mean_sorted_indices = np.argsort(mean_distances)
mean_indices = mean_sorted_indices[:3]

print(f"\nKAN Loss Statistics:")
print(f"Mean Loss: {mean_loss:.6f}")
print(f"Min Loss: {individual_losses[lowest_indices[0]]:.6f}")
print(f"Max Loss: {individual_losses[highest_indices[-1]]:.6f}")


Analyzing individual KAN prediction losses...

KAN Loss Statistics:
Mean Loss: 0.000613
Min Loss: 0.000000
Max Loss: 0.093456


In [7]:
print("\nAnalyzing individual MLP prediction losses...")
mlp_individual_losses = []
mlp_predictions = []

mlp_model.eval()
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = mlp_model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        mlp_individual_losses.append(loss)
        mlp_predictions.append(prediction)

mlp_individual_losses = np.array(mlp_individual_losses)
mlp_predictions = torch.stack(mlp_predictions)
mlp_sorted_indices = np.argsort(mlp_individual_losses)

mlp_mean_loss = np.mean(mlp_individual_losses)
mlp_lowest_indices = mlp_sorted_indices[:3] 
mlp_highest_indices = mlp_sorted_indices[-3:]

mlp_mean_distances = np.abs(mlp_individual_losses - mlp_mean_loss)
mlp_mean_sorted_indices = np.argsort(mlp_mean_distances)
mlp_mean_indices = mlp_mean_sorted_indices[:3]

print(f"\nMLP Loss Statistics:")
print(f"Mean Loss: {mlp_mean_loss:.6f}")
print(f"Min Loss: {mlp_individual_losses[mlp_lowest_indices[0]]:.6f}")
print(f"Max Loss: {mlp_individual_losses[mlp_highest_indices[-1]]:.6f}")


Analyzing individual MLP prediction losses...

MLP Loss Statistics:
Mean Loss: 0.000670
Min Loss: 0.000000
Max Loss: 0.008651


In [8]:
table_data = []
categories = [("Lowest", lowest_indices), ("Highest", highest_indices), ("Mean", mean_indices)]

for label, indices in categories:
    for idx in indices:
        input_sample = X_test[idx]
        input_str = f"({input_sample[0]:.2f}, {input_sample[1]:.2f}, ..., {input_sample[-1]:.2f})"
        table_data.append({
            "Category": label,
            "Index": idx,
            "Input (Summary)": input_str,
            "True Value": y_test[idx].item(),
            "Predicted": predictions[idx].item(),
            "Loss": individual_losses[idx]
        })

df_kan_analysis = pd.DataFrame(table_data)
df_kan_analysis

,Category,Index,Input (Summary),True Value,Predicted,Loss
0,Lowest,312,"(0.81, 0.75, ..., 0.29)",1.670678,1.670690,1.566747e-10
1,Lowest,724,"(0.10, -0.40, ..., 0.72)",1.671821,1.671837,2.785328e-10
2,Lowest,793,"(-0.60, -0.08, ..., -0.85)",1.745200,1.745229,8.117382e-10
3,Highest,762,"(-0.12, 0.73, ..., 0.07)",1.478509,1.309722,2.848897e-02
4,Highest,1139,"(0.25, 0.30, ..., -0.15)",1.866362,1.691866,3.044887e-02
5,Highest,611,"(0.02, 0.05, ..., 0.02)",1.429473,1.123768,9.345552e-02
6,Mean,747,"(0.78, 0.17, ..., 0.28)",1.621798,1.597072,6.114121e-04
7,Mean,975,"(-0.90, -0.64, ..., 0.21)",1.710931,1.686216,6.108227e-04
8,Mean,822,"(-0.56, 0.40, ..., 0.24)",1.726556,1.701764,6.146529e-04


In [9]:
table_data_mlp = []
categories = [("Lowest", mlp_lowest_indices), ("Highest", mlp_highest_indices), ("Mean", mlp_mean_indices)]

for label, indices in categories:
    for idx in indices:
        input_sample = X_test[idx]
        input_str = f"({input_sample[0]:.2f}, {input_sample[1]:.2f}, ..., {input_sample[-1]:.2f})"
        table_data_mlp.append({
            "Category": label,
            "Index": idx,
            "Input (Summary)": input_str,
            "True Value": y_test[idx].item(),
            "Predicted": mlp_predictions[idx].item(),
            "Loss": mlp_individual_losses[idx]
        })

df_mlp_analysis = pd.DataFrame(table_data_mlp)
df_mlp_analysis

,Category,Index,Input (Summary),True Value,Predicted,Loss
0,Lowest,1289,"(-0.71, -0.45, ..., 0.67)",1.659051,1.659092,1.730896e-09
1,Lowest,737,"(0.28, 0.18, ..., 0.71)",1.648283,1.648221,3.783726e-09
2,Lowest,372,"(0.75, 0.14, ..., 0.62)",1.666398,1.666504,1.135783e-08
3,Highest,473,"(-0.21, 0.75, ..., 0.31)",1.680355,1.597175,6.918924e-03
4,Highest,340,"(0.93, 0.33, ..., 0.22)",1.757890,1.674672,6.925292e-03
5,Highest,706,"(0.71, 0.18, ..., 0.42)",1.573648,1.666660,8.651317e-03
6,Mean,1343,"(0.64, 0.52, ..., 0.69)",1.633411,1.607524,6.701498e-04
7,Mean,829,"(0.51, -0.51, ..., 0.90)",1.585027,1.559168,6.686816e-04
8,Mean,143,"(0.05, 0.40, ..., 0.82)",1.603037,1.577200,6.675601e-04


In [10]:
mlp_save_path = "model_pkls/functionexp100_mlp_model.pkl"
torch.save({
    'model_state_dict': mlp_model.state_dict(),
    'config': {
        'input_dim': 100,
        'hidden_dims': [64, 64, 64, 64],
        'output_dim': 1
    }
}, mlp_save_path)
print(f"MLP Model saved to {mlp_save_path}")

MLP Model saved to model_pkls/functionexp100_mlp_model.pkl


In [11]:
kan_save_path = "model_pkls/functionexp100_kan_model.pkl"
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'layers_hidden': [100, 1, 1],
        'grid_min': -1,
        'grid_max': 1,
        'num_grids': 10,
        'use_base_update': False,
        'use_layernorm': False,
    }
}, kan_save_path)
print(f"KAN Model saved to {kan_save_path}")

KAN Model saved to model_pkls/functionexp100_kan_model.pkl
